In [1]:
# imports

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [2]:
# load trained model

model_path = "./emotion_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [3]:
# emotion labels

emotion_labels = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization",
    "relief","remorse","sadness","surprise","neutral"
]

In [4]:
# improved predict emotion (hybrid)

def predict_emotion(text, threshold=0.25):
    text_lower = text.lower()

    # rule-based hints (boost accuracy)
    if "sad" in text_lower or "bad" in text_lower:
        return [("sadness", 0.9)]
    if "happy" in text_lower:
        return [("joy", 0.9)]
    if "angry" in text_lower or "hate" in text_lower:
        return [("anger", 0.9)]
    if "all good" in text_lower or "fine" in text_lower or "okay" in text_lower:
    return [("neutral", 0.9)]

    # fallback to model
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits)[0]

    idx = torch.argmax(probs).item()

    return [(emotion_labels[idx], probs[idx].item())]

In [5]:
import random

def generate_response(emotions, user_input):
    
    primary = emotions[0][0]

    responses = {
        "sadness": [
            "I'm really sorry you're feeling this way. What happened?",
            "That sounds tough. Do you want to talk about it?",
            "I'm here for you. Tell me more."
        ],
        "joy": [
            "That's awesome! 😊 What made you feel this way?",
            "Nice! I love hearing that.",
            "Sounds like a good moment!"
        ],
        "anger": [
            "I understand you're upset. What’s going on?",
            "That sounds frustrating. Want to vent?",
            "Take a breath — I’m listening."
        ],
        "fear": [
            "That sounds stressful. What are you worried about?",
            "You're not alone. Want to share more?",
            "I’m here. Tell me what’s bothering you."
        ],
        "neutral": [
            f"You said: '{user_input}'. Tell me more about that.",
            "Interesting. Can you explain a bit more?",
            "I’m listening — go on.",
            "Hmm, what do you mean by that?"
        ]
    }

    return random.choice(responses.get(primary, ["I'm here to listen."]))
    

In [6]:
# response bank

response_bank = {
    "sadness": [
        "I'm really sorry you're feeling this way. What happened?",
        "That sounds tough. Do you want to talk about it?",
        "I'm here for you. Tell me more."
    ],
    "joy": [
        "That's awesome! 😊 What made you feel this way?",
        "Nice! I love hearing that.",
        "Sounds like a good moment!"
    ],
    "anger": [
        "I understand you're upset. What’s going on?",
        "That sounds frustrating. Want to vent?",
        "Take a breath — I’m listening."
    ],
    "fear": [
        "That sounds stressful. What are you worried about?",
        "You're not alone. Want to share more?",
        "I’m here. Tell me what’s bothering you."
    ],
    "approval": [
        "That’s good to hear.",
        "Sounds like things are okay.",
        "Nice, glad things are fine."
    ],
    "neutral": [
        "Tell me more.",
        "I'm listening.",
        "Go on.",
        "What do you mean?"
    ]
}

In [7]:
# generate response

import random

def generate_response(emotions, user_input):
    primary = emotions[0][0]

    responses = response_bank.get(primary, ["I'm here to listen."])

    return random.choice(responses)

In [8]:
# improved safety check

def safety_check(text):
    emotions = predict_emotion(text)

    for emo, score in emotions:
        if emo in ["anger", "disgust", "annoyance"] and score > 0.3:
            return True

    return False

In [ ]:
# chatbot loop

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "bye", "quit"]:
        print("Bot: Take care! 👋")
        break

    if safety_check(user_input):
        print("Bot: Let's keep things respectful.")
        print()
        continue

    emotions = predict_emotion(user_input)
    response = generate_response(emotions, user_input)

    print("Emotion:", emotions)
    print("Bot:", response)
    print()
    

You:  i am feeling sad


Emotion: [('sadness', 0.9)]
Bot: I'm here for you. Tell me more.



You:  hello


Emotion: [('neutral', 0.11194135993719101)]
Bot: Tell me more.



You:  i am feeling lazy


Emotion: [('neutral', 0.19765080511569977)]
Bot: What do you mean?



You:  i am feeling shy


Emotion: [('neutral', 0.12374115735292435)]
Bot: Go on.



You:  epstein 


Emotion: [('neutral', 0.6053443551063538)]
Bot: What do you mean?

